# Preparing Data for a model to forecast the country's total energy consumption

As the title suggests, in this notebook, I intend to focus on preparing data for the model.

I am beginning the load forecasting task immediately after the successful implementation of an offshore wind power forecasting model, as I believe that features derived from the load data could be useful when developing more complex models later on.

In my current approach, I am operating on the premise that the load is, to a large extent, dependent on calendar-based features.

**However, the plan is to try incorporating features such as temperature or storm warnings into the set of predictors, as these can affect heating costs or, theoretically, lead to the suspension of operations and a drop in load.**

In [31]:
# Imports
import pandas as pd
import numpy as np
import os
import pyrootutils
import holidays
from dotenv import load_dotenv

# Needed to load variables from the .env file into the environment
load_dotenv()

# Determining the project root based on the .gitignore file
PROJECT_ROOT = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)
from utils.weather import fetch_load_weather

# Determining data directories
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw_data" / "load_model"
CLEAN_DATA_DIR = PROJECT_ROOT / "data" / "clean_data" / "load_model"
DATASETS_DIR = PROJECT_ROOT / "data" / "datasets" / "load_model"

### In the cell below, I'm creating the basic structure for the future dataset.
For now, I'll use `UTC time` as the index, as it's unique and will be useful for joins.

For the first column, I've used converted `Berlin time`, this will be used to calculate the temporal features for the models.

The step size is 15 minutes, as the target (`load_act`) has that resolution.

In [32]:
# Determining the time interval for the energy load data
START = '2024-03-01'
END = '2026-06-20'

# Creating a DataFrame with UTC and Berlin timestamps
# Index (UTC Timestamps)
utc_index = pd.date_range(
    start=START, 
    end=END, 
    freq='15min', 
    tz='UTC'
)

# Converting UTC timestamps to Berlin time
berlin_time = utc_index.tz_convert('Europe/Berlin')

# Combining both UTC and Berlin timestamps into a single DataFrame
df = pd.DataFrame({
    'dt_berlin': berlin_time
}, index=utc_index)

# Displaying a sample of the DataFrame to verify its structure
df.sample(5)  # Displaying a sample of the DataFrame to verify its structure

,dt_berlin
2025-03-23 19:15:00+00:00,2025-03-23 20:15:00+01:00
2026-04-21 11:45:00+00:00,2026-04-21 13:45:00+02:00
2024-11-27 00:15:00+00:00,2024-11-27 01:15:00+01:00
2024-10-23 06:45:00+00:00,2024-10-23 08:45:00+02:00
2026-01-29 01:45:00+00:00,2026-01-29 02:45:00+01:00


## Preparing Calendar Features
In this cell, I prepare calendar-based features. These include:
- `time_sin`, `time_cos` - local time **cyclically** encoded for continuity
- `quarter_index` - an **experimental** feature that shows the number of 15-minute intervals within a day, calculated from 0:00 **local** time, may be useful for boosting models
- `doy` - an **experimental** feature that shows the day of the year; may also be useful
- `doy_sin`, `doy_cos` - **continuously** encoded day of the year
- `holiday_index` - a **weighted** holiday index. To create the weight, the regions where the holiday takes place are multiplied by the annual total energy consumption of that region according to the Monitoringbericht 2025 report, after which it is normalized as a percentage from 0.0 to 1.0
- `workday_index` - an **experimental** flag indicating whether it is a workday, it excludes weekends and `holiday_index`
- `is_monday` ... `is_sunday` - **binary** flags indicating what day of the week it is

In [33]:
# Encoding the time of day using sine and cosine
local = df['dt_berlin']
minutes_from_midnight = local.dt.hour * 60 + local.dt.minute
time_phase = 2 * np.pi * minutes_from_midnight / 1440

df["time_sin"] = np.sin(time_phase)
df["time_cos"] = np.cos(time_phase)


# Creating a 15-minute day index
quarter_index = minutes_from_midnight // 15
df['quarter_index'] = quarter_index


# Adding a column for the day of the year based on Berlin time
df['doy'] = local.dt.dayofyear

# Encoding the day of the year using sine and cosine
# Calculating the number of days in the year for each date, accounting for leap years
days_in_year = 365 + df["dt_berlin"].dt.is_leap_year.astype(int)
doy_phase = 2 * np.pi * (df["doy"] - 1) / days_in_year

df["doy_sin"] = np.sin(doy_phase)
df["doy_cos"] = np.cos(doy_phase)

# Adding a column for the day of the week based on Berlin time
df['dow'] = local.dt.day_name()


# Adding a column for the holiday weigted index
# Information taken from:
# Monitoringbericht 2025 - Monitoringbericht gemäß § 63 Abs. 3 i. V. m. § 35 EnWG und § 48 Abs. 3 i. V. m. § 53 Abs. 3 GWB
# https://data.bundesnetzagentur.de/Bundesnetzagentur/SharedDocs/Mediathek/Monitoringberichte/MonitoringberichtEnergie2025.pdf
# Pages 96-97
# Consumption volumes (TWh) by states at the VNB level for 2024 (Bundesnetzagentur)
consumption_twh = {
    'NW': 86.94,  # Nordrhein-Westfalen
    'BY': 63.02,  # Bayern
    'BW': 52.09,  # Baden-Württemberg
    'HE': 40.45,  # Hessen
    'NI': 35.58,  # Niedersachsen
    'SN': 19.81,  # Sachsen
    'RP': 17.97,  # Rheinland-Pfalz
    'ST': 13.48,  # Sachsen-Anhalt
    'BE': 12.31,  # Berlin
    'BB': 11.59,  # Brandenburg
    'SH': 11.22,  # Schleswig-Holstein
    'TH': 10.05,  # Thüringen
    'HH': 9.91,   # Hamburg
    'MV': 6.06,   # Mecklenburg-Vorpommern
    'SL': 5.07,   # Saarland
    'HB': 4.47    # Bremen
}

# Normalizing the consumption volumes to get weights for each state
total_twh = sum(consumption_twh.values())
state_weights = {state: val / total_twh for state, val in consumption_twh.items()}


# Creating a column for the weighted holiday indicator based on the state weights and holidays
dates = pd.to_datetime(df['dt_berlin'].dt.date)
years = dates.dt.year.unique().tolist()

# Creating a dictionary of holiday calendars for each state in Germany for the relevant years
state_calendars = {
    state: holidays.Germany(subdiv=state, years=years)
    for state in state_weights
}

# Calculating the weight only for unique dates
unique_dates = dates.dt.date.unique()
date_to_weight = {
    d: sum(weight for state, weight in state_weights.items() if d in state_calendars[state])
    for d in unique_dates
}

# Mapping the calculated weights to the original DataFrame based on the date
df['holiday_index'] = dates.dt.date.map(date_to_weight)

# Creating a boolean mask to identify weekdays (Monday to Friday)
is_weekday = (local.dt.dayofweek < 5)

# Creating a new column 'workday_index' that combines the weekday and holiday information
df["workday_index"] = (
    is_weekday.astype(float)
    * (1.0 - df["holiday_index"])
)

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Encoding the day of the week into one-hot encoded columns
encoded = pd.get_dummies(
    pd.Categorical(df["dow"], categories=days),
    dtype="int8"
).set_axis([f"is_{day.lower()}" for day in days], axis=1)

df = pd.concat([df.drop(columns="dow"), encoded.set_axis(df.index)], axis=1)

# Displaying a sample of the DataFrame to verify its structure
df.sample(5)

# Saving the cleaned and combined weather data to a parquet file in the clean data directory
# WARNING: Does not overwrite if already exists
min_cal_day = df.index.min()
max_cal_day = df.index.max()

CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

clean_path = CLEAN_DATA_DIR / (
    f"load_calendar_features_{min_cal_day.date().strftime('%Y-%m-%d')}"
    f"_to_{max_cal_day.date().strftime('%Y-%m-%d')}.parquet"
)

if not clean_path.exists():
    df.to_parquet(clean_path, index=False)
    print(f"Saved: {clean_path}")
else:
    print(f"File already exists, skipping: {clean_path}")

# "quarter_index", "doy", "workday_index"
# These signs are still candidates, they may contain valuable information for boosting, but this requires verification

Saved: D:\python_projects\entsoe_2nd_try\data\clean_data\load_model\load_calendar_features_2024-03-01_to_2026-06-20.parquet


### Weather from individual forecast runs

The load profile uses land cells: temperature at 2 m (°C), wind speed at 10 m (m/s), shortwave radiation (W/m², mean over the preceding hour), and cloud cover (%). The grid targets about 20 locations; it is spatially uniform, not consumption-weighted. Freeze the coordinates alongside the raw data before comparing models.

For each Berlin delivery day, use the previous day's 00:00 UTC run. The fetcher retains one buffer hour on each side, including DST days. Keep `datetime_run_utc` when joining or interpolating: adjacent runs deliberately overlap in their buffer hours. The returned data is hourly and has not yet been joined to the quarter-hour load grid.

## Preparing geographic coordinates
I need to prepare N equidistant points covering the entire area of ​​Germany to collect weather data for them. For this, I'll use my utility.

In [5]:
from utils.germany_grid_map import generate_germany_grid, visualize_grid

# Use the repository's boundary snapshot for reproducible coordinates.
load_points = generate_germany_grid(
    n_points=20,
    states_url=(PROJECT_ROOT / 'utils/germany_states_low.geojson').as_uri(),
)
load_points

# Creates an .html file with a map and point locations
# visualize_grid(load_points)

[germany_grid] Loading federal state boundaries...
[germany_grid] Finding grid spacing for about 20 points...
[germany_grid] Grid spacing: 140.27 km
[germany_grid] Points inside Germany: 20
[germany_grid] Assigning points to federal states...
[germany_grid] Done. Total points: 20


{1: [8.935644, 48.358116, 'Baden-Württemberg'],
 2: [10.82771, 48.360139, 'Bayern'],
 3: [12.718808, 48.329824, 'Bayern'],
 4: [7.945168, 49.43719, 'Rheinland-Pfalz'],
 5: [9.879022, 49.455853, 'Baden-Württemberg'],
 6: [11.813026, 49.44134, 'Bayern'],
 7: [6.90947, 50.506619, 'Rheinland-Pfalz'],
 8: [8.886832, 50.542796, 'Hessen'],
 9: [10.865672, 50.544926, 'Thüringen'],
 10: [12.843362, 50.513005, 'Sachsen'],
 11: [7.84762, 51.620384, 'Nordrhein-Westfalen'],
 12: [9.873275, 51.640069, 'Niedersachsen'],
 13: [11.899109, 51.62476, 'Sachsen-Anhalt'],
 14: [13.922253, 51.574493, 'Brandenburg'],
 15: [8.832033, 52.72618, 'Niedersachsen'],
 16: [10.90829, 52.728431, 'Sachsen-Anhalt'],
 17: [12.98317, 52.694702, 'Brandenburg'],
 18: [9.866803, 53.823428, 'Schleswig-Holstein'],
 19: [11.996039, 53.80722, 'Mecklenburg-Vorpommern'],
 20: [8.770126, 54.909071, 'Schleswig-Holstein']}

## Downloading Weather Data
In this cell, I download weather data using my own utility. It loads the forecast single run after midnight UTC on day D-1 for delivery day D.
The resulting dataframe has the following structure:
- `datetime_run_utc` - the UTC date when the weather model run was performed, needed for navigation and debugging
- `datetime_utc` - the UTC date of the delivery period on day D, useful for navigation and joins
- `datetime_berlin` - the date converted to local time, initially needed for feature engineering, but already completed in the previous step, can be used for debugging
- `point_id` - the ID of the geographic point (probe) from which the weather data was taken, needed for feature engineering and weather feature aggregation, as well as for debugging
- `region` - the federal state in which the point is located, needed for debugging
- `latitude`, `longitude` - geographic latitude and longitude indicators, needed primarily for debugging
- `temperature_2m` - the outside temperature, **critical** information for Models
- `wind_speed_10m` - wind speed, can be very useful for accounting for the heating/cooling needs of buildings
- `shortwave_radiation` - sunlight, can be useful for accounting for the heating/cooling needs of buildings
- `cloud_cover` - an **experimental** feature indicating cloud cover, partially duplicated in `shortwave_radiation`, but may have additional predictive power

**Open Meteo Historical Forecast** provides 5,000 API requests per hour and 10,000 per day for free. One request equals one single run for one geographic point, the number of features does not affect the query weight.

In [6]:
# Define the time interval for fetching weather data
WEATHER_START = '2026-06-25'
WEATHER_END = '2026-11-17'

# WeatherSTART/WeatherEND below are interpreted as inclusive Berlin delivery dates for weather.
WEATHER_RUN_START = (pd.Timestamp(WEATHER_START) - pd.DateOffset(days=1)).strftime('%Y-%m-%d')
WEATHER_RUN_END = (pd.Timestamp(WEATHER_END) - pd.DateOffset(days=1)).strftime('%Y-%m-%d')
FETCH_LOAD_WEATHER = False  # Enable explicitly for the historical download.

# 20 coordinates with a 3-second interval fit within the minute API-limits
if FETCH_LOAD_WEATHER:
    import importlib
    from utils import weather as weather_module

    weather_module = importlib.reload(weather_module)
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

    temp_checkpoint = RAW_DATA_DIR / '_tmp_load_ECMWF_singleruns.parquet'
    print(f'Weather module: {weather_module.__file__}')
    print(f'Temp checkpoint: {temp_checkpoint.resolve()}')

    load_weather = weather_module.fetch_load_weather(
        WEATHER_RUN_START,
        WEATHER_RUN_END,
        points=load_points,
        checkpoint_path=temp_checkpoint,
    )

    dt_utc = pd.to_datetime(load_weather['datetime_utc'], utc=True)
    actual_start = dt_utc.min().strftime('%Y-%m-%d')
    actual_end = dt_utc.max().strftime('%Y-%m-%d')

    stem = f'load_ECMWF_singleruns_{actual_start}_to_{actual_end}'
    final_parquet = RAW_DATA_DIR / f'{stem}.parquet'

    load_weather.to_parquet(final_parquet, index=False)
    if temp_checkpoint.exists() and temp_checkpoint != final_parquet:
        temp_checkpoint.unlink()

    pd.DataFrame.from_dict(
        load_points, orient='index', columns=['longitude', 'latitude', 'region']
    ).rename_axis('point_id').to_json(RAW_DATA_DIR / f'{stem}_points.json', orient='table')

    print(f'Saved parquet: {final_parquet.resolve()}')

In [7]:
# Checking the structure of the fetched weather data
if FETCH_LOAD_WEATHER:
    display(load_weather.sample(10))
else:
    display(pd.read_parquet( RAW_DATA_DIR / 'load_ECMWF_singleruns_2025-07-24_to_2026-06-23.parquet').sample(10))

,datetime_run_utc,datetime_utc,datetime_berlin,point_id,region,latitude,longitude,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover
141412,2025-10-09 00:00:00+00:00,2025-10-10 21:00:00+00:00,2025-10-10 23:00:00+02:00,17,Brandenburg,52.688927,12.953272,12.60,4.924429,0.0,100.0
17018,2026-06-07 00:00:00+00:00,2026-06-08 11:00:00+00:00,2026-06-08 13:00:00+02:00,2,Bayern,48.330402,10.854271,24.40,1.627882,863.0,92.0
49423,2026-03-05 00:00:00+00:00,2026-03-06 20:00:00+00:00,2026-03-06 21:00:00+01:00,6,Bayern,49.455181,11.772805,3.95,1.990603,0.0,0.0
109593,2026-02-03 00:00:00+00:00,2026-02-04 00:00:00+00:00,2026-02-04 01:00:00+01:00,13,Sachsen-Anhalt,51.634445,11.945454,-2.65,3.703039,0.0,100.0
73587,2025-12-20 00:00:00+00:00,2025-12-21 04:00:00+00:00,2025-12-21 05:00:00+01:00,9,Thüringen,50.579964,10.831859,2.85,1.345362,0.0,96.0
21923,2026-01-12 00:00:00+00:00,2026-01-13 02:00:00+00:00,2026-01-13 03:00:00+01:00,3,Bayern,48.330402,12.663317,1.80,2.163331,0.0,97.0
58533,2026-03-21 00:00:00+00:00,2026-03-22 04:00:00+00:00,2026-03-22 05:00:00+01:00,7,Rheinland-Pfalz,50.509663,6.837456,1.70,1.978004,0.0,3.0
137115,2026-03-28 00:00:00+00:00,2026-03-29 14:00:00+00:00,2026-03-29 16:00:00+02:00,16,Sachsen-Anhalt,52.688927,10.934580,10.95,5.135173,519.0,36.0
14892,2026-03-17 00:00:00+00:00,2026-03-18 17:00:00+00:00,2026-03-18 18:00:00+01:00,2,Bayern,48.330402,10.854271,8.45,4.776243,99.0,0.0
40243,2026-02-15 00:00:00+00:00,2026-02-16 18:00:00+00:00,2026-02-16 19:00:00+01:00,5,Baden-Württemberg,49.455181,9.913941,2.35,5.255949,0.0,100.0


## Checking the accuracy of geographic points.
Checking the accuracy of geographic points.
After loading, I noticed that the `latitude` and `longitude` parameters were slightly different from those requested.

The error is **small** and **shouldn't affect** the result for my task.

The point is that after transmitting the requested coordinates, Open Meteo converts them to its own grid. Therefore, just in case, I decided to check whether the coordinates were different between data collection sessions.

In [8]:
import json

files = [
    RAW_DATA_DIR / "load_ECMWF_singleruns_2026-06-24_to_2026-11-17_points.json",
    RAW_DATA_DIR / "load_ECMWF_singleruns_2024-11-17_to_2025-07-25_points.json",
    RAW_DATA_DIR / "load_ECMWF_singleruns_2024-03-14_to_2024-11-19_points.json",
]

loaded = [json.loads(p.read_text(encoding="utf-8")) for p in files]

print(
    "1 and 2 are identical:", loaded[0] == loaded[1]
)
print(
    "1 and 3 are identical:", loaded[0] == loaded[2]
)
print(
    "2 and 3 are identical:", loaded[1] == loaded[2]
)

1 and 2 are identical: True
1 and 3 are identical: True
2 and 3 are identical: True


## Filling the missing ECMWF run on 23 June 2026

The 00:00 UTC run on 23 June returns missing `temperature_2m` and `wind_speed_10m` values for all load points on the Berlin delivery day of 24 June. For this single exception, I use the longer forecast horizon of the 22 June ECMWF run for **all four** weather variables.

The prepared rows keep `datetime_run_utc = 2026-06-23` as the nominal run date expected by the existing daily dataset. `source_run_utc` records the real forecast run, and `is_fallback_run` marks the substitution. The source data is not simulated; only the run identifier is assigned to the missing daily slot.

The Berlin delivery day has 24 hourly timestamps, plus one buffer hour on each side (26 timestamps for this summer date). The result is saved separately from the raw download and its checkpoint, so the missing original run can still be revisited later.

In [9]:
# Defining the missing nominal run and the available source run
from utils import weather as weather_module

NOMINAL_RUN_UTC = pd.Timestamp('2026-06-23', tz='UTC')
SOURCE_RUN_UTC = pd.Timestamp('2026-06-22', tz='UTC')
DELIVERY_DAY = pd.Timestamp('2026-06-24')

# Building the Berlin delivery window with one physical hour on each side
delivery_start_utc = DELIVERY_DAY.tz_localize('Europe/Berlin').tz_convert('UTC')
delivery_end_utc = (DELIVERY_DAY + pd.DateOffset(days=1)).tz_localize('Europe/Berlin').tz_convert('UTC')
expected_hours = pd.date_range(
    delivery_start_utc - pd.Timedelta(hours=1),
    delivery_end_utc,
    freq='h',
)

# Keeping this exceptional, nominally dated run separate from the raw archive
fallback_path = RAW_DATA_DIR / 'load_ECMWF_fallback_nominal_run_2026-06-23.parquet'

if fallback_path.exists():
    fallback_weather = pd.read_parquet(fallback_path)
else:
    client = weather_module._build_client()
    params = {
        'latitude': [coords[1] for coords in load_points.values()],
        'longitude': [coords[0] for coords in load_points.values()],
        'run': SOURCE_RUN_UTC.strftime('%Y-%m-%dT%H:%M'),
        'hourly': list(weather_module.LOAD_VARIABLES),
        'wind_speed_unit': 'ms',
        'cell_selection': 'land',
        'models': 'ecmwf_ifs',
        'forecast_days': 4,
    }
    responses = client.weather_api(weather_module.SINGLE_RUNS_URL, params=params, timeout=60)
    if len(responses) != len(load_points):
        raise ValueError(f'Expected {len(load_points)} point responses, got {len(responses)}.')

    frames = []
    for (point_id, coords), response in zip(load_points.items(), responses):
        hourly = weather_module._parse_response(response, list(weather_module.LOAD_VARIABLES))
        hourly = hourly.loc[hourly['datetime_utc'].isin(expected_hours)].copy()
        if not pd.DatetimeIndex(hourly['datetime_utc']).equals(expected_hours):
            raise ValueError(f'Incomplete 24 June hourly window for point {point_id}.')
        if not np.isfinite(hourly[list(weather_module.LOAD_VARIABLES)].to_numpy(dtype=float)).all():
            raise ValueError(f'Missing weather values in the 22 June source run for point {point_id}.')

        hourly['datetime_run_utc'] = NOMINAL_RUN_UTC
        hourly['datetime_berlin'] = hourly['datetime_utc'].dt.tz_convert('Europe/Berlin')
        hourly['point_id'] = point_id
        hourly['region'] = coords[2]
        hourly['latitude'] = response.Latitude()
        hourly['longitude'] = response.Longitude()
        # hourly['source_run_utc'] = SOURCE_RUN_UTC
        hourly['is_fallback_run'] = True
        frames.append(hourly[weather_module.LOAD_OUTPUT_COLS + [
            # 'source_run_utc',
            'is_fallback_run']])

    fallback_weather = pd.concat(frames, ignore_index=True)

# Checking the saved result too, including reruns that reuse the file
key = ['datetime_run_utc', 'datetime_utc', 'point_id']
if (len(fallback_weather) != len(load_points) * len(expected_hours)
        or fallback_weather.duplicated(key).any()
        or set(fallback_weather['point_id']) != set(load_points)
        or not fallback_weather['datetime_run_utc'].eq(NOMINAL_RUN_UTC).all()
        # or not fallback_weather['source_run_utc'].eq(SOURCE_RUN_UTC).all()
        or not fallback_weather['is_fallback_run'].eq(True).all()
        or not np.isfinite(fallback_weather[list(weather_module.LOAD_VARIABLES)].to_numpy(dtype=float)).all()
        or any(not pd.DatetimeIndex(fallback_weather.loc[fallback_weather['point_id'].eq(point_id), 'datetime_utc'])
               .as_unit('ns').sort_values().equals(expected_hours.as_unit('ns'))
               for point_id in load_points)):
    raise ValueError('The fallback run failed its coverage or provenance checks.')

if not fallback_path.exists():
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    fallback_weather.to_parquet(fallback_path, index=False)

print(f'Saved/loaded fallback: {fallback_path.resolve()}')
print(f'{len(load_points)} points × {len(expected_hours)} hours = {len(fallback_weather)} rows')
display(fallback_weather.head())

Saved/loaded fallback: D:\python_projects\entsoe_2nd_try\data\raw_data\load_model\load_ECMWF_fallback_nominal_run_2026-06-23.parquet
20 points × 26 hours = 520 rows


,datetime_run_utc,datetime_utc,datetime_berlin,point_id,region,latitude,longitude,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover,is_fallback_run
0,2026-06-23 00:00:00+00:00,2026-06-23 21:00:00+00:00,2026-06-23 23:00:00+02:00,1,Baden-Württemberg,48.330402,8.894472,22.299999,1.614001,0.0,3.0,True
1,2026-06-23 00:00:00+00:00,2026-06-23 22:00:00+00:00,2026-06-24 00:00:00+02:00,1,Baden-Württemberg,48.330402,8.894472,21.799999,0.424264,0.0,2.0,True
2,2026-06-23 00:00:00+00:00,2026-06-23 23:00:00+00:00,2026-06-24 01:00:00+02:00,1,Baden-Württemberg,48.330402,8.894472,20.850000,0.300000,0.0,2.0,True
3,2026-06-23 00:00:00+00:00,2026-06-24 00:00:00+00:00,2026-06-24 02:00:00+02:00,1,Baden-Württemberg,48.330402,8.894472,20.000000,0.552268,0.0,2.0,True
4,2026-06-23 00:00:00+00:00,2026-06-24 01:00:00+00:00,2026-06-24 03:00:00+02:00,1,Baden-Württemberg,48.330402,8.894472,19.400000,0.750000,0.0,1.0,True


## Saving Cleaned and Concatenated Data
In this notebook, I concatenate **all** the weather data for the load model, **including** the missing reconstructed run on `June 23, 2026`.
Afterwards, I save it to the clean data directory.

In [34]:
# Setting up paths for the raw weather data files
path1 = RAW_DATA_DIR / 'load_ECMWF_singleruns_2024-03-14_to_2024-11-19.parquet'
path2 = RAW_DATA_DIR / 'load_ECMWF_singleruns_2024-11-17_to_2025-07-25.parquet'
path3 = RAW_DATA_DIR / 'load_ECMWF_singleruns_2025-07-24_to_2026-06-23.parquet'
path4 = RAW_DATA_DIR / 'load_ECMWF_singleruns_2026-06-24_to_2026-11-17.parquet'
fallback_path = RAW_DATA_DIR / 'load_ECMWF_fallback_nominal_run_2026-06-23.parquet'

# Combining all the weather data files into a single DataFrame, including the fallback run for 23 June 2026
list_of_paths = [path1, path2, path3, path4, fallback_path]

df_complete = pd.concat([pd.read_parquet(path) for path in list_of_paths], ignore_index=True)\
    .sort_values(by=['datetime_run_utc', 'datetime_utc', 'point_id']).reset_index(drop=True)

# Filling any missing values in the 'is_fallback_run' column with False
df_complete['is_fallback_run'] = df_complete['is_fallback_run'].fillna(False)


key_cols = ['datetime_run_utc', 'datetime_utc', 'point_id']
dups = df_complete[df_complete.duplicated(keep=False)]

# Checking duplicates by delivery date
dup_days_forecast = (
    pd.to_datetime(dups['datetime_utc'])
    .dt.normalize()
    .drop_duplicates()
    .sort_values()
)

# Duplicates are expected at the junctions of files, they can be safely deleted.
print(f'Found {len(dup_days_forecast)} days with duplicate forecasts')
print(dup_days_forecast)

# Dropping duplicates, keeping the last occurrence for each combination of 'datetime_run_utc', 'datetime_utc', and 'point_id'
df_complete = df_complete.drop_duplicates(keep='last').reset_index(drop=True)

# Checking for any remaining duplicates after dropping them
print(df_complete[df_complete.duplicated(subset=key_cols, keep=False)].shape[0], 'duplicate rows remain after dropping duplicates')


# Checking for missing days in the combined weather data
actual_days = pd.DatetimeIndex(
    pd.to_datetime(df_complete['datetime_run_utc']).dt.tz_convert('UTC').dt.normalize().unique()
).sort_values()

min_day = actual_days.min()
max_day = actual_days.max()
expected_days = pd.date_range(start=min_day, end=max_day, freq='D', tz='UTC')
missing_days = expected_days.difference(actual_days)

# Printing the summary of the period, expected days, actual days, and any missing days
print(f"\n\nDays range: {min_day.date()} -> {max_day.date()}")
print(f"Expected days: {len(expected_days)}, Found: {len(actual_days)}")
print(f"Missing days: {len(missing_days)}")

if len(missing_days) > 0:
    print("Missing dates:")
    print(missing_days)


# Saving the cleaned and combined weather data to a parquet file in the clean data directory
# WARNING: Does not overwrite if already exists
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

clean_path = CLEAN_DATA_DIR / (
    f"load_ECMWF_weather_{min_day.date().strftime('%Y-%m-%d')}"
    f"_to_{max_day.date().strftime('%Y-%m-%d')}.parquet"
)

if not clean_path.exists():
    df_complete.to_parquet(clean_path, index=False)
    print(f"Saved: {clean_path}")
else:
    print(f"File already exists, skipping: {clean_path}")

Found 6 days with duplicate forecasts
128960   2024-11-17 00:00:00+00:00
129040   2024-11-18 00:00:00+00:00
130080   2024-11-19 00:00:00+00:00
258940   2025-07-23 00:00:00+00:00
259060   2025-07-24 00:00:00+00:00
260100   2025-07-25 00:00:00+00:00
Name: datetime_utc, dtype: datetime64[ms, UTC]
0 duplicate rows remain after dropping duplicates


Days range: 2024-03-14 -> 2026-11-16
Expected days: 978, Found: 978
Missing days: 0
File already exists, skipping: D:\python_projects\entsoe_2nd_try\data\clean_data\load_model\load_ECMWF_weather_2024-03-14_to_2026-11-16.parquet
